In [1]:
# Note sulla versione: questo notepad è stato fatto con un ambiente conda con python 3.6 per la compatibilità con anc2vec.
# Per riprodurre: https://github.com/sinc-lab/anc2vec
# 
# Questo notepad è dedicato alla fase di data pre-processing antecedente all'elaborazione in GNN
# Passaggi:
#   1. Generazione matrice di adiacenza (o altra rappresentazione compatibile con GNN)
#   2. Generazione matrice delle feature
#   2.1     Encoding delle GO

In [2]:
# Qui verranno ottenuti gli embedding per le GO presenti nel grafo da analizzare
import pandas as pd
import numpy as np
import anc2vec

embeddings = anc2vec.get_embeddings()

In [51]:
import torch

# Apro i file dei nodi e degli archi
nodes_file = '../files/networks/union_nodes.csv'
nodes_df = pd.read_csv(nodes_file)[['name', 'Gene Ontology IDs']].dropna(subset=['Gene Ontology IDs']) # N.B. c'è una proteina senza GO
node_ids = {node: i for i, node in enumerate(nodes_df['name'].tolist())}

edges_file = '../files/networks/union_edges.csv'
raw_edges_df = pd.read_csv(edges_file)['name']

# Ottengo una lista di go e li associo agli embedding
unique_go_ids = set(
    id.strip() for ids in nodes_df['Gene Ontology IDs'].dropna().str.split(';') for id in ids
)
unique_go_ids_list = list(unique_go_ids)

# NOTA: questa è una prova, vengono saltati i termini che il modello non conosce (andrebbe ri addestrato)
go_to_embedding = {go:embeddings[go] for go in unique_go_ids if go in embeddings}

def merge_embeddings(go_list):
    embeddings = [go_to_embedding[g.strip()] for g in go_list if g in go_to_embedding]
    return np.mean(embeddings, axis=0).tolist()

# Sostituisco i nodi con degli id
node_ids = {node: i for i, node in enumerate(nodes_df['name'].tolist())}
edges_df = raw_edges_df.str.extract(r'(\S+) \(interacts with\) (\S+)')
edges_df.columns = ['node1', 'node2']

edges_df['node1'] = edges_df['node1'].map(node_ids)
edges_df['node2'] = edges_df['node2'].map(node_ids)

edges_df = edges_df[~edges_df["node2"].isna()] # Una proteina non è annotata, da aggiustare dopo aver fatto il prototipo

# Definisco il tensore degli archi (ogni riga indica un arco)
edge_index = torch.tensor(edges_df[["node1", "node2"]].values, dtype=torch.long).t().contiguous()

# Sostituisco gli id dei nodi con quelli generati prima
nodes_df['name'] = nodes_df['name'].map(node_ids)

# Aggrega tutte le go di ogni record
nodes_df['Gene Ontology IDs'] = nodes_df['Gene Ontology IDs'].dropna().apply(
    lambda x: merge_embeddings([go.strip() for go in x.split(';')])
)
features = []
for c in nodes_df['Gene Ontology IDs'].dropna():
    features.append(c)

# Crea il vettore di feature
x = torch.tensor(features, dtype=torch.float)  # Esclude "Entry Name"
print(edges_df)

     node1  node2
0        0    1.0
2        1  104.0
3        1   27.0
4        1   96.0
5        1   79.0
..     ...    ...
661    153  156.0
662    153  140.0
663    153  159.0
664    153  144.0
665    153  143.0

[663 rows x 2 columns]


In [52]:
# Qui serializzo le strutture necessarie al training della GNN
# Questo perché questo notepad è python3.6 per compatibilità con anc2vec, PyG invece vuole python >= 3.9

torch.save({
    'edge_index': edge_index,
    'node_features': x,
    'node_ids': node_ids
}, './model_output/pre_processed_data.pt')